# 01. Онлайн-детекторы на БВС-5 и НСУ-3: конденсат (угроза №28) и метеоусловия (угрозы №5, 7, 8, 9)

**Данные только из ТЗ:** БВС-5 (`temperature`, `humidity`, `wind_speed`, `wind_gust`, `power_consumption`; 1 Гц, шум датчиков, потери ≈ 3 %, задержка p99 до 0,9 с) и НСУ-3 (`visibility_m`, `precipitation_prob`, `turbulence_index`; ≈ 0,16 Гц, данные старше 30 с устарели). Всё считается **потоково, за O(1) на сообщение**; потоки приходят с джиттером, потерями и переупорядочиванием (стенд ноутбука 00).

## Что решаем
| Часть | Угроза | Вход | Выход |
|---|---|---|---|
| A | 28 Конденсат | БВС-5 (T, RH) | предупреждение «запотевание объектива вероятно» (без повторов) |
| B | 5 Облачность/туман, 7 Дождь, 8 Снег, 9 Град (как метео-prior), аэрозоль (1, 14, 15) | БВС-5 + НСУ-3 | класс условий: `dry / rain / snow / fog_cloud / aerosol / hail` |

## Главная проблема онлайн-режима
Шум датчиков (σ_T = 0,3 К, σ_RH = 2 %) при пороговом решении даёт **«дребезг»**: десятки алертов на одно событие и ложные тревоги. Поэтому каждый детектор получает слой подавления повторов: сглаживание, подтверждение (n подряд), гистерезис и паузу после отбоя (`src/alerts.py`). Параметры подбираются на отдельных сценариях, оценка идёт на других зёрнах.

> **Честность оценки.** Сценарии синтетические: связи погоды с полями заданы генератором (ноутбук 00), поэтому классы разделяются легче, чем в реальности. Для этого есть абляции (без НСУ-3, смена условий и калибровки датчиков). Реальных данных по БПЛА нет.

In [ ]:
import sys, pathlib, warnings, time
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT)); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

from src.eval_bvs5 import (KINDS, build_cache, eval_condensation, build_rows, dwell_states, events_from_alerts, segments, label_at)
from src.detectors_bvs5 import (CondensationV2, ConditionDetector, NumpyLogReg, SkWrap, FEATS, FEATS_BVS5, CLASSES)
from src.metrics import ece, reliability_curve
from src.streams import DT
plt.rcParams["figure.dpi"] = 110; pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

## Часть A. Конденсат на объективе

### Метод (из прежнего ноутбука 1, но потоково)
Температура объектива не измеряется, поэтому оценивается звеном первого порядка по температуре воздуха из БВС-5: `T̂_L' = (T_a + Δ̂ − T̂_L)/τ̂`; точка росы по Магнусу `T_d(T, RH)`; запас `D̂ = T_d − T̂_L` (сглаживается EMA); вероятность `p = Φ((D̂ + m)/σ)`. Слой подавления повторов: включение при `p ≥ 0,6` **n_on** сообщений подряд и если прошла **пауза** после прошлого отбоя; выключение при `p ≤ 0,3` n_off подряд.

### Оценка
Истина - скрытая физика линзы из генератора (τ и Δ случайны и детектору неизвестны). Событие - интервал `lens_fog` (разрывы < 30 с объединяются). Корректная тревога - в окне [onset − 120 с; конец + 30 с]. Метрики: **recall**, **lead** (сколько секунд до onset, медиана), **алертов на событие**, **ложных тревог в час** (тревоги вне окон по всем сценариям).

In [ ]:
t0 = time.time()
tune = build_cache(KINDS, range(0, 15)); test = build_cache(KINDS, range(100, 115))
print("сценариев: подбор", len(tune), "| тест", len(test), "| время", round(time.time() - t0, 1), "с")
pos = {k: sum(1 for c in tune if c["kind"] == k and c["truth"]["lens_fog"].any()) for k in KINDS}
print("сценариев с событием lens_fog в выборке подбора:", {k: v for k, v in pos.items() if v})
rows = []
for name, mk in (("без подавления, Δ̂=0", lambda: CondensationV2(offset=0.0)), ("без подавления, Δ̂=−1", lambda: CondensationV2())):
    rows.append(dict(вариант=name, выборка="подбор", **eval_condensation(mk, tune))); rows.append(dict(вариант=name, выборка="тест", **eval_condensation(mk, test)))
base = pd.DataFrame(rows).set_index(["вариант", "выборка"]).round(2); display(base)

Простое пороговое решение даёт полноту 1,0, но **десятки алертов на событие** и **десятки ложных тревог в час**: в таком виде оно непригодно. Случайный поиск по параметрам подавления на выборке подбора (160 комбинаций: Δ̂, m, сглаживание α, n_on, n_off, пауза, τ̂); выбираем максимальную полноту при ограничениях `FA ≤ 5/ч` и `алертов на событие ≤ 3`.

In [ ]:
rng = np.random.default_rng(0); grid = dict(offset=[-1.5, -1.0, -0.5, 0.0], margin=[0.0, 0.5, 1.0], alpha=[1.0, 0.3, 0.1], n_on=[1, 3, 5], n_off=[3, 10, 20], cooldown=[0, 60, 120], tau=[90, 120, 180])
rows = []
for i in range(160):
    p = {k: v[rng.integers(len(v))] for k, v in grid.items()}
    rows.append({**p, **eval_condensation(lambda: CondensationV2(**p), tune)})
search = pd.DataFrame(rows)
ok = search[(search.fa_per_hour <= 5) & (search.alerts_per_event <= 3)].sort_values(["recall", "fa_per_hour"], ascending=[False, True])
print("допустимых конфигураций:", len(ok), "из", len(search)); display(ok.head(5).round(2))
BEST = {k: float(ok.iloc[0][k]) for k in grid}; BEST["n_on"] = int(BEST["n_on"]); BEST["n_off"] = int(BEST["n_off"])
print("выбрано:", BEST)
fig, ax = plt.subplots(figsize=(6.2, 4)); sc = ax.scatter(search.fa_per_hour, search.recall, c=np.log10(search.alerts_per_event.clip(lower=1)), cmap="viridis", s=18)
ax.set_xscale("symlog", linthresh=1); ax.set_xlabel("ложных тревог в час (symlog)"); ax.set_ylabel("recall"); plt.colorbar(sc, label="log10(алертов на событие)"); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
rows = []
for name, mk in (("без подавления (Δ̂=0)", lambda: CondensationV2(offset=0.0)), ("с подавлением (подобрано)", lambda: CondensationV2(**BEST))):
    rows.append(dict(вариант=name, **eval_condensation(mk, test)))
cmp_ = pd.DataFrame(rows).set_index("вариант").round(2); display(cmp_)
# устойчивость: потери БВС-5 и смещение датчиков (калибровка)
rows = []
for label, kw in (("потери 3 % (номинал)", dict(loss5=None)), ("потери 10 %", dict(loss5=0.10)), ("потери 30 %", dict(loss5=0.30)), ("потери 50 %", dict(loss5=0.50)),
                  ("смещение T +0,5 К, RH +4 %", dict(bias=(0.5, 4.0))), ("смещение T −0,5 К, RH −4 %", dict(bias=(-0.5, -4.0)))):
    ch = build_cache(KINDS, range(200, 208), streams=("BVS5",), **kw)
    rows.append(dict(условие=label, **eval_condensation(lambda: CondensationV2(**BEST), ch)))
rob = pd.DataFrame(rows).set_index("условие").round(2); display(rob)

In [ ]:
def n_alerts(c_):
    d = CondensationV2(offset=0.0); d.reset(); return sum(len(d.on_message(m)) for m in c_["msgs"] if m["stream"] == "BVS5")
cands = [i for i, c_ in enumerate(test) if c_["truth"]["lens_fog"].any()]
ci = max(cands, key=lambda i: n_alerts(test[i]))            # иллюстрация: сценарий с наибольшим «дребезгом» базового детектора
c_ = test[ci]; res = {}; print("сценарий:", c_["kind"], "seed", c_["seed"])
for name, mk in (("без подавления", lambda: CondensationV2(offset=0.0)), ("с подавлением", lambda: CondensationV2(**BEST))):
    d = mk(); d.reset(); ts, ps, al = [], [], []
    for m in c_["msgs"]:
        if m["stream"] == "BVS5":
            a = d.on_message(m)
            if getattr(d, "last_prob", None) is not None: ts.append(m["t_gen"]); ps.append(d.last_prob)
            al += [m["t_gen"] for _ in a]
    res[name] = (np.array(ts), np.array(ps), al)
fig, axs = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)
axs[0].plot(c_["S"]["t"], c_["S"]["T"], label="T воздуха"); axs[0].plot(c_["S"]["t"], c_["S"]["Td"], label="T_d"); axs[0].plot(c_["S"]["t"], c_["S"]["T_lens"], label="T линзы (скрыта)"); axs[0].legend(ncol=3, fontsize=8)
for a, (name, (t, p, al)) in zip(axs[1:], res.items()):
    a.plot(t, p, lw=.7); a.vlines(al, 0, 1, color="r", lw=1.2, label=f"алертов: {len(al)}"); a.set_ylabel(f"p, {name}"); a.legend(loc="upper left", fontsize=8)
for a in axs:
    for (s0, s1) in segments(c_["truth"]["lens_fog"]): a.axvspan(s0, s1, color="g", alpha=.12)
    a.grid(alpha=.3)
axs[2].set_xlabel("t, с (зелёное - истинное запотевание)"); plt.tight_layout(); plt.show()

## Часть B. Классификатор метеоусловий (БВС-5 + НСУ-3)

### Признаки (потоково, окно 30 с по `timestamp`)
| Признаки | Источник |
|---|---|
| `T, RH, spread = T − T_d, T_w` (Stull), `wind, gust, gust − wind` | БВС-5 |
| EMA(30 с) для `T, RH, spread`, наклоны за 30 с `dT, dRH, dspread`, EMA мощности | БВС-5 |
| `ln visibility_m`, `precipitation_prob`, `turbulence_index`, возраст прогноза | НСУ-3 |

Классы (приоритет при совпадении): `hail > snow > rain > fog_cloud > aerosol (пыль, дым) > dry`. Метка - скрытое состояние на момент генерации сообщения. Модели: правило (T_w, RH, spread), многоклассовая логрегрессия (чистый numpy на борту), градиентный бустинг. Обучение и тест - разные зёрна сценариев. Затем подтверждение смены класса (dwell) для подавления «мерцания» состояния.

In [ ]:
t0 = time.time()
train_c = build_cache(KINDS, range(300, 320)); test_c = build_cache(KINDS, range(400, 412))
shift = lambda cfg: (setattr(cfg, "wind0", cfg.wind0 * 1.5), setattr(cfg, "V0", 12000.0))
shift_c = build_cache(KINDS, range(500, 508), bias=(0.5, 4.0), shift=shift)
Xtr, ytr, _ = build_rows(train_c); Xte, yte, mte = build_rows(test_c); Xsh, ysh, msh = build_rows(shift_c)
print("строк: обучение", Xtr.shape, "| тест", Xte.shape, "| смена условий", Xsh.shape, "| время", round(time.time() - t0), "с")
print(pd.Series(ytr).value_counts().to_dict())
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, class_weight="balanced")).fit(Xtr, ytr)
hg = HistGradientBoostingClassifier(max_iter=150, random_state=0).fit(Xtr, ytr)
hg5 = HistGradientBoostingClassifier(max_iter=150, random_state=0).fit(Xtr[:, :14], ytr)
lr5 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, class_weight="balanced")).fit(Xtr[:, :14], ytr)
def rule(X):
    p = np.full(len(X), "dry", dtype=object)
    p[(X[:, 1] >= 93) & (X[:, 3] < 1.0)] = "snow"; p[(X[:, 1] >= 93) & (X[:, 3] >= 1.0)] = "rain"
    p[X[:, 2] < 0.15] = "fog_cloud"; p[(X[:, 1] < 40) & (X[:, 6] > 2.5)] = "aerosol"; return p
models = {"правило": (lambda X: rule(X)), "LogReg (БВС-5+НСУ-3)": lr.predict, "HGB (БВС-5+НСУ-3)": hg.predict,
          "LogReg (только БВС-5)": lambda X: lr5.predict(X[:, :14]), "HGB (только БВС-5)": lambda X: hg5.predict(X[:, :14])}
rows = []
for name, f in models.items():
    r = {"модель": name}
    for dn, (X, y) in (("тест", (Xte, yte)), ("смена условий", (Xsh, ysh))):
        p = f(X); r[f"macroF1 {dn}"] = f1_score(y, p, average="macro"); r[f"acc {dn}"] = accuracy_score(y, p)
    rows.append(r)
cls_tab = pd.DataFrame(rows).set_index("модель").round(3); display(cls_tab)

In [ ]:
labels = CLASSES
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
for a, (name, f) in zip(axs, (("HGB (БВС-5+НСУ-3)", models["HGB (БВС-5+НСУ-3)"]), ("HGB (только БВС-5)", models["HGB (только БВС-5)"]))):
    cm = confusion_matrix(yte, f(Xte), labels=labels, normalize="true"); a.imshow(cm, vmin=0, vmax=1, cmap="Blues")
    a.set_xticks(range(6), labels, rotation=45, ha="right", fontsize=8); a.set_yticks(range(6), labels, fontsize=8); a.set_title(name, fontsize=9)
    for (i, j), v in np.ndenumerate(cm): a.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="k" if v < .6 else "w")
plt.tight_layout(); plt.show()
rs = np.random.default_rng(0); idx = rs.choice(len(Xte), 6000, replace=False)
pi = permutation_importance(hg, Xte[idx], yte[idx], n_repeats=3, random_state=0, scoring="f1_macro")
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values(); imp.plot.barh(figsize=(5.5, 4.4)); plt.xlabel("падение macro-F1"); plt.tight_layout(); plt.show()

In [ ]:
# калибровка вероятности «идут осадки» (rain ∪ snow ∪ hail)
yprec = np.isin(yte, ["rain", "snow", "hail"]); fig, ax = plt.subplots(figsize=(4.8, 4.2)); rows = []
for name, m, X in (("LogReg", lr, Xte), ("HGB", hg, Xte)):
    P = m.predict_proba(X); cl = list(m.classes_); pp = sum(P[:, cl.index(k)] for k in ("rain", "snow", "hail"))
    x, y, _ = reliability_curve(yprec, pp); ax.plot(x, y, "o-", label=f"{name}, ECE={ece(yprec, pp):.3f}"); rows.append((name, ece(yprec, pp)))
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("предсказанная вероятность осадков"); ax.set_ylabel("доля осадков"); ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

### Событийная оценка и подавление «мерцания» состояния
Строка классификатора приходит каждую секунду; без подтверждения класс может «мерцать». Смена состояния принимается после **dwell** одинаковых предсказаний подряд. Метрики по классам: recall (тревога нужного класса в окне [onset − 30 с; конец + 30 с]), медианная задержка, ложные тревоги в час, **переключений состояния в час**. Перебор dwell делается офлайн по предсказаниям (проверено ниже: результат совпадает с онлайн-детектором).

In [ ]:
def per_scenario(model_predict, X, meta, n):
    cls = model_predict(X); by = {i: ([], []) for i in range(n)}
    for (ci, t), c_ in zip(meta, cls): by[ci][0].append(c_); by[ci][1].append(t)
    return by
def sweep(name, f, X, meta, cache):
    by = per_scenario(f, X, meta, len(cache)); rows = []
    for dw in (1, 2, 3, 5, 8):
        trans = [dwell_states(by[i][0], by[i][1], dw) for i in range(len(cache))]
        ev, sw = events_from_alerts(cache, trans)
        rows.append(dict(модель=name, dwell=dw, recall_средний=(ev.recall * ev.событий).sum() / ev.событий.sum(), ложных_в_час=ev.ложных_в_час.sum(), переключений_в_час=sw,
                         задержка_медиана_с=ev.задержка_медиана_с.median()))
    return rows
sw = pd.DataFrame(sweep("HGB", models["HGB (БВС-5+НСУ-3)"], Xte, mte, test_c) + sweep("LogReg", models["LogReg (БВС-5+НСУ-3)"], Xte, mte, test_c)
                  + sweep("HGB только БВС-5", models["HGB (только БВС-5)"], Xte, mte, test_c)).round(2)
display(sw.set_index(["модель", "dwell"]))

In [ ]:
DW = 3
by = per_scenario(models["HGB (БВС-5+НСУ-3)"], Xte, mte, len(test_c))
trans = [dwell_states(by[i][0], by[i][1], DW) for i in range(len(test_c))]; ev_full, sw_full = events_from_alerts(test_c, trans)
by5 = per_scenario(models["HGB (только БВС-5)"], Xte, mte, len(test_c))
ev_b5, sw_b5 = events_from_alerts(test_c, [dwell_states(by5[i][0], by5[i][1], DW) for i in range(len(test_c))])
print(f"HGB, dwell={DW}: переключений в час = {sw_full:.1f}"); display(ev_full.round(2))
print(f"HGB только БВС-5, dwell={DW}: переключений в час = {sw_b5:.1f}"); display(ev_b5.round(2))
# проверка: офлайн-перебор == онлайн-детектор
d_ok = True
for i in range(0, len(test_c), 20):
    d = ConditionDetector(SkWrap(hg), dwell=DW); d.reset(); tr = []; prev = d.state
    for m in test_c[i]["msgs"]:
        if m["stream"] in d.streams:
            d.on_message(m)
            if d.state != prev: tr.append((m["t_gen"], d.state)); prev = d.state
    d_ok &= (tr == trans[i])
print("онлайн-детектор совпадает с офлайн-подсчётом:", d_ok)

## Время обработки
Замер `perf_counter_ns` на сообщение (включая извлечение признаков и предсказание). Бортовой запас оценивается как при ноутбуке ×50. Требование: быстрый контур ≲ 5 мс на сообщение; БВС-5 приходит 1 Гц.

In [ ]:
def timing(make, cache, n_scen=6):
    ts = []
    for c_ in cache[:n_scen]:
        d = make(); d.reset()
        for m in c_["msgs"]:
            if m["stream"] in d.streams:
                t1 = time.perf_counter_ns(); d.on_message(m); ts.append((time.perf_counter_ns() - t1) / 1e3)
    a = np.array(ts); return dict(сообщений=len(a), p50_мкс=np.percentile(a, 50), p99_мкс=np.percentile(a, 99), max_мкс=a.max())
rows = [dict(детектор="CondensationV2", **timing(lambda: CondensationV2(**BEST), test)),
        dict(детектор="Condition + LogReg (numpy)", **timing(lambda: ConditionDetector(NumpyLogReg(lr), dwell=DW), test_c)),
        dict(детектор="Condition + HGB (sklearn, построчно)", **timing(lambda: ConditionDetector(SkWrap(hg), dwell=DW), test_c))]
tm = pd.DataFrame(rows).set_index("детектор").round(1); tm["p99_мс при замедлении ×50"] = (tm.p99_мкс * 50 / 1000).round(2); display(tm)

## Выходной интерфейс
Алерты оформляются по полям БВС-3 (в ТЗ нет типа алерта «погода»: подтип кодируется в `description`, `alert_type` оставлен под согласование).

In [ ]:
def condensation_alert(t, p, uav_id="UAV-001"):
    return {"alert_id": f"fog-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": 2, "description": "LENS_FOG_WARNING", "sensor_source": "BVS5:temperature+humidity",
            "confidence": round(float(p), 2), "timestamp": int(t * 1000), "recommended_action": "lower_camera_confidence; avoid_fast_descent_into_humid_layer"}
def condition_alert(t, cls, p, uav_id="UAV-001"):
    sev = {"rain": 2, "snow": 3, "fog_cloud": 3, "aerosol": 3, "hail": 4}[cls]
    act = {"rain": "lower_lidar_confidence; request_weather(БВС-10)", "snow": "lower_lidar_confidence; check_power_residual", "fog_cloud": "lower_camera_confidence; consider_altitude_change",
           "aerosol": "lower_camera_confidence; monitor_cpu_temp", "hail": "emergency_land_or_RTH; log_impacts"}[cls]
    return {"alert_id": f"wx-{int(t * 1000)}", "uav_id": uav_id, "alert_type": None, "severity": sev, "description": "WX_" + cls.upper(), "sensor_source": "BVS5+NSU3",
            "confidence": round(float(p), 2), "timestamp": int(t * 1000), "recommended_action": act}
display(condensation_alert(412.0, 0.83)); display(condition_alert(300.0, "snow", 0.91))

## 8. Выводы и ограничения
**Часть A: конденсат (только БВС-5).**
- Без подавления повторов детектор даёт **≈ 28 алертов на событие и ≈ 14 ложных тревог в час** (при полноте 1,0). Слой подавления (сглаживание, подтверждение, пауза после отбоя) на независимых сценариях даёт **≈ 1,5 алерта на событие и ≈ 2 ложные тревоги в час** при полноте 0,93; медианное упреждение ≈ 20 с до начала запотевания. Это главный результат части A: качество определяется логикой подавления, а не вычислениями.
- Устойчивость: потери БВС-5 до 50 % почти не сказываются (полнота ≥ 0,97). **Смещение калибровки датчиков (±0,5 К и ±4 % RH) заметно снижает качество** (полнота 0,79-0,83, при отрицательном смещении тревога приходит с опозданием на ≈ 39 с). Для реальной эксплуатации нужна калибровка и мониторинг дрейфа датчиков T/RH.
- Физическая причина ограничения прежняя: температура объектива не измеряется, параметры τ̂ и Δ̂ подобраны, а не известны.

**Часть B: метеоусловия (БВС-5 + НСУ-3).**
- Градиентный бустинг по признакам потоков: macro-F1 0,94 на тесте; на **смене условий** (смещение датчиков, ветер ×1,5, фоновая дымка) **0,71**; без НСУ-3 только 0,76 на тесте и **0,46** на смене условий. Правило по RH, T_w и spread: 0,41 и 0,33; логрегрессия: 0,86 и 0,66.
- **Оговорка о честности.** Наиболее важные признаки - `precipitation_prob` и `visibility_m` из НСУ-3, которые генератор строит из скрытой истины с шумом, а также T и T_w, где диапазоны температуры привязаны к типу сценария. Поэтому оценки с НСУ-3 оптимистичны; реалистичнее ориентироваться на результат «только БВС-5» и на смену условий. Реальные данные METAR (ноутбук-предшественник) и лётные логи необходимы для подтверждения.
- Подтверждение смены класса (dwell = 3 сообщения, ≈ 3 с) снижает «мерцание» состояния с 41,6 до 14,5 переключений в час и ложные тревоги с 7,7 до 1,65 в час при сохранении полноты 1,0 и медианной задержке 3,7 с. Для бустинга без НСУ-3 полнота по дождю 0,83 при 2,35 ложных в час.
- Калибровка вероятности осадков: см. диаграмму надёжности (ECE приведены в легенде); использовать вероятности в модуле доверия следует после калибровки.

**Время.** Конденсат: ≈ 1 мкс на сообщение (p99 ≈ 2 мкс). Классификатор на логрегрессии (чистый numpy, с извлечением признаков): p50 ≈ 25 мкс, p99 ≈ 49 мкс. Бустинг построчно через sklearn: p50 ≈ 0,76 мс, p99 ≈ 1,6 мс (при замедлении ×50 ≈ 79 мс) - укладывается в период БВС-5 (1 с), но **не в быстрый контур ≲ 5 мс при большом замедлении**. Для бортового исполнения бустинг нужно экспортировать в компактный вид (≤ 100 деревьев, векторный код) или вызывать классификатор асинхронно: метеоусловия медленные, и в критический путь 200 мс он не входит.

**Ограничения.** Синтетический генератор (связи погоды с полями - наши допущения); класс `hail` отличается от `rain` почти только порывами и турбулентностью; тип `aerosol` объединяет пыль и дым (различить их по этим полям нельзя); пауза и подтверждение увеличивают задержку (3-10 с для классификатора).

**Дальше:** ноутбук 02 (фантомы БВС-4: расхождение lidar / radar / camera как признак осадков, пыли и тумана и как индикатор слепоты датчиков), затем мощность и устойчивость (налипание, град), деградация камеры и слияние.